# ◆ LeadHarbour: from a score to a decision

Run `leadharbour fetch` first, from the repository root. The official UCI data describes bank contacts from 2008 to 2010. It is useful for learning, not for choosing current recipients. The outcome is a response label, not a causal uplift label.

In [ ]:
from pathlib import Path
from leadharbour.data import FEATURES, FORBIDDEN_FEATURES, load_dataset
from leadharbour.model import ordered_split, train

source = Path('data/bank-additional-full.csv')
frame = load_dataset(source)
training, validation, test = ordered_split(frame)
print('Rows:', len(frame), 'Features:', FEATURES)
print('Outcome rates:', [round(part.y.eq('yes').mean(), 3) for part in (training, validation, test)])
assert set(FEATURES).isdisjoint(FORBIDDEN_FEATURES)


The original rows are ordered by date. Keeping a later period for testing exposes a change in outcome rate that a random split could hide. Call duration is excluded because it is known only after a call. Age and several demographic fields are also excluded from the baseline, though the remaining fields still need a fairness review.

In [ ]:
report = train(source, Path('artifacts/model.joblib'))
print('Validation:', report['validation'])
print('Later test period:', report['test'])
assert report['test']['rows'] > 0


A ranking measure is not enough. Compare average precision with the outcome rate, inspect the Brier score, and ask whether the probability means the same thing in a later period. The model card records the source hash and split so a result can be reproduced.

## Compare models without changing the later test period

The comparison uses one fixed set of parameters for each candidate. It trains only on the first period. The baseline always predicts the training outcome rate. Logistic regression and XGBoost use the same five input fields. A second logistic model adds a flag calculated only from `previous`. These runs do not replace the deployed model artifact. On macOS, XGBoost may require `brew install libomp`.


In [ ]:
from leadharbour.comparison import compare_models

comparison = compare_models(source)
print("Source shifts:", comparison["profiles"])
for model_name, periods in comparison["results"].items():
    print(model_name)
    for period_name, metrics in periods.items():
        print(period_name, {name: metrics.get(name) for name in (
            "roc_auc", "average_precision", "brier_score", "log_loss",
            "precision_at_10_percent", "ties_at_cutoff",
            "precision_at_10_percent_bounds")})


The training positive rate is much lower than the later test rate. Previous contacts and previous successes also become far more common. This means that a model can have a reasonable ranking score while its probability estimates are poor, or the reverse. Compare average precision with the period base rate, inspect Brier and log loss, and check whether the top ten percent is actually useful for a fixed review capacity. The constant baseline has no meaningful top-ten ranking because all its scores tie. The other models also tie many records at the review cutoff. Their top-ten precision is the expected value if tied records are picked at random, and the possible bounds show how uncertain that choice is.


In [ ]:
baseline = comparison["results"]["training_rate"]["later_test"]
for name in ("logistic_regression", "logistic_with_history_flag", "xgboost"):
    result = comparison["results"][name]["later_test"]
    print(name, "ROC AUC", result["roc_auc"],
          "Brier", result["brier_score"],
          "tie-aware precision at 10%", result["precision_at_10_percent"],
          "ties", result["ties_at_cutoff"])
print("Later-period positive rate:", baseline["positive_rate"])


The derived history flag is a feature-engineering experiment, not a new deployment field. It is largely redundant with `previous` and `poutcome` here. Any feature must be available at prediction time, justified for the use case, derived with the same rule in training and serving, and checked for drift and fairness. A flexible model can fit an old campaign more closely and still rank the next one worse. Treat this single comparison as diagnosis, not hyperparameter tuning on the test period.


## Thresholds and calibration tell different stories

The validation period is where a threshold or review capacity would be chosen. A 0.5 cutoff is just a convention; this model predicts much lower values. The 0.05 and 0.06 cutoffs are demonstrations, not approved decisions. The later-period bins below are for inspection only: predicted probability and observed rate should be close if a model is well calibrated for that period. Small or sparse bins deserve caution.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix
from leadharbour.data import feature_frame
from leadharbour.model import load_artifact

model = load_artifact(Path("artifacts/model.joblib"))["pipeline"]
validation_probabilities = model.predict_proba(feature_frame(validation))[:, 1]
validation_labels = validation.y.eq("yes").astype(int)
for threshold in (0.05, 0.06, 0.5):
    tn, fp, fn, tp = confusion_matrix(
        validation_labels, validation_probabilities >= threshold, labels=[0, 1]
    ).ravel()
    print("Validation threshold", threshold, "TP", tp, "FP", fp, "FN", fn, "TN", tn)

test_probabilities = model.predict_proba(feature_frame(test))[:, 1]
observed, predicted = calibration_curve(
    test.y.eq("yes").astype(int), test_probabilities, n_bins=5, strategy="quantile"
)
print("Later period: mean prediction, observed rate")
for expected, actual in zip(predicted, observed):
    print(round(expected, 3), round(actual, 3))


A threshold trades false positives against false negatives. A reliability table shows whether the numeric probabilities travel well to the later period. Neither one tells us whether contacting somebody caused a conversion. This data has no no-contact control group.


In [ ]:
from leadharbour.decision import Candidate, expected_net_value, rank_for_review

synthetic = [Candidate('example-a', 0.20, 100, 5),
             Candidate('example-b', 0.05, 100, 5)]
print([(item.lead_id, expected_net_value(item)) for item in synthetic])
print('For human review:', [item.lead_id for item in rank_for_review(synthetic, 1)])


Expected net value is `probability × value if converted − contact cost`. The numbers above are invented to explain the arithmetic. A response probability does not tell us the extra conversions caused by contact. For that, we need a credible treatment and control comparison, with consent and business rules checked separately.